In [ ]:
import os
os.environ['PYTHONHASHSEED'] = '50'
os.environ['TF_DETERMINISTIC_OPS'] = '1'

from google.colab import drive
import pandas as pd
import numpy as np

import tensorflow as tf
EfficientNetB0 = tf.keras.applications.EfficientNetB0
preprocess_input = tf.keras.applications.efficientnet.preprocess_input
Dense = tf.keras.layers.Dense
Dropout = tf.keras.layers.Dropout
Model = tf.keras.models.Model
SGD = tf.keras.optimizers.SGD
ModelCheckpoint = tf.keras.callbacks.ModelCheckpoint
EarlyStopping = tf.keras.callbacks.EarlyStopping
ReduceLROnPlateau = tf.keras.callbacks.ReduceLROnPlateau
ImageDataGenerator = tf.keras.preprocessing.image.ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import random
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report

tf.keras.utils.set_random_seed(50)

In [ ]:
# connection to google drive
drive.mount('/content/drive')
print(os.listdir('/content/drive/MyDrive'))

MODEL_DIR = '/content/drive/MyDrive/skripsi_models_testing'
os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
# read dataset
csv_path = '/content/drive/MyDrive/Dataset_Pakaian_Final.csv'
df = pd.read_csv(csv_path)

print(df.shape)

In [ ]:
# read the image dataset
image_path = '/content/drive/MyDrive/Dataset'

# combining the image path from local to drive
def getImagePath(local_path, base_dir):
    filename = os.path.basename(local_path)

    real_file_name = os.path.join(base_dir, filename)
    if os.path.exists(real_file_name):
        return real_file_name
    else:
      return None

df['image_full_path'] = df['image_local_path'].apply(lambda p: getImagePath(p, image_path))

# checking the result
found_mask = df['image_full_path'].notna()
print(f"Total baris yang ada di dataset: {len(df)}")
print(f"Total : {found_mask.sum()}")
print(f"Tidak ditemukan: {(~found_mask).sum()}")

In [ ]:
# create new dataset
df_clean = df[df['image_full_path'].notna()].copy().reset_index(drop=True)
print(f"Dataset final: {len(df_clean)} baris, {df_clean['category'].nunique()} kategori unik")

In [ ]:
category_counts = df_clean['category'].value_counts()
print(f"Jumlah kategori yang dimiliki: {len(category_counts)}")
print(category_counts)

print(f"\nKategori dengan sampel paling sedikit:")
print(category_counts.tail(10))

In [ ]:
# checking for missing value
tmp = df_clean.copy()

tmp = tmp.dropna(subset=['category'])
print("Setelah apus mising value:", tmp.shape)

tmp['category'] = tmp['category'].astype(str).str.lower()

counts = tmp['category'].value_counts()
tmp = tmp[tmp['category'].isin(counts[counts >= 7].index)]
print("\nHasil akhir    :", tmp.shape)

df_clean = tmp.copy()

In [ ]:
# # use 50% of the data for code testing

# FRACTION = 0.5
# MIN_PER_CLASS = 7
# def sample_per_class(g):
#     n = max(MIN_PER_CLASS, int(round(len(g) * FRACTION)))
#     return g.sample(n=min(n, len(g)), random_state=50)

# print("Sebelum:", len(df_clean))
# # have to group by it first because apply works per column not rows
# df_clean = (df_clean.groupby('category', group_keys=False)
#                     .apply(sample_per_class)
#                     .reset_index(drop=True))
# print("Sesudah:", len(df_clean))
# print(df_clean['category'].value_counts())

In [ ]:
le = LabelEncoder()
df_clean['category_encoded'] = le.fit_transform(df_clean['category'])

print("Mapping kategori:")
for i, cat in enumerate(le.classes_):
    print(f"{i}: {cat}")

SEED = 50

# Split dataset 70:15:15
train_df, temp_df = train_test_split(
    df_clean,
    test_size=0.30,
    stratify=df_clean['category'],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['category'],
    random_state=SEED
)

print(f"\nTrain: {len(train_df)} baris")
print(f"Val  : {len(val_df)} baris")
print(f"Test : {len(test_df)} baris")

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# data augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest',
)

# preprocessing
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='image_full_path',
    y_col='category',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=list(le.classes_),
    shuffle=True,
    seed=SEED,
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    x_col='image_full_path',
    y_col='category',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=list(le.classes_),
    shuffle=False,
)

test_generator = datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='image_full_path',
    y_col='category',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=list(le.classes_),
    shuffle=False,
)

x_batch, y_batch = next(train_generator)
print("Shape gambar per batch:", x_batch.shape)
print("Shape label per batch :", y_batch.shape)
print("Contoh label (one-hot):", y_batch[0])
print("Kategori asli baris pertama batch ini:", le.classes_[y_batch[0].argmax()])

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x_batch[i].astype('uint8'))
    plt.title(le.classes_[y_batch[i].argmax()])
    plt.axis('off')
plt.suptitle('Contoh citra latih setelah augmentasi')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'contoh_augmentasi.png'), dpi=150)
plt.show()

In [ ]:
base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3),
    pooling='avg'
)

for layer in base_model.layers:
    layer.trainable = False

UNFREEZE_FROM = ('block5', 'block6', 'block7', 'top_')
for layer in base_model.layers:
    if layer.name.startswith(UNFREEZE_FROM):
        layer.trainable = True

trainable_count = sum(l.trainable for l in base_model.layers)
frozen_count = len(base_model.layers) - trainable_count
print(f"Total layer base model: {len(base_model.layers)}")
print(f"Trainable: {trainable_count}")
print(f"Frozen: {frozen_count}")

In [ ]:
x = base_model.output
x = Dropout(0.3)(x)
outputs = Dense(len(le.classes_), activation='softmax', name='classification_head')(x)

model = Model(inputs=base_model.input, outputs=outputs)
model.summary()

In [ ]:
model.compile(
    optimizer=SGD(learning_rate=0.0005, momentum=0.9),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
class_labels = np.unique(train_generator.classes)
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=class_labels,
    y=train_generator.classes
)
class_weight_dict = dict(zip(class_labels, class_weights_array))
print(class_weight_dict)

In [ ]:
checkpoint_path = os.path.join(MODEL_DIR, 'best_model.keras')

callbacks = [
    ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
]

EPOCHS = 45
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy per Epoch (dengan augmentasi)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss per Epoch (dengan augmentasi)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Loss    : {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

In [ ]:
df_clean.to_csv(os.path.join(MODEL_DIR, 'df_clean.csv'), index=False)

In [ ]:
feature_extractor = Model(
    inputs=base_model.input,
    outputs=base_model.output,
    name='feature_extractor'
)
feature_extractor.summary()

feature_extractor.save(os.path.join(MODEL_DIR, 'feature_extractor.keras'))

In [ ]:
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix — Test Set (dengan augmentasi)')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
from IPython.display import display

EVAL_DIR = os.path.join(MODEL_DIR, 'evaluasi_retrieval')
os.makedirs(EVAL_DIR, exist_ok=True)

if 'feature_extractor' not in globals():
    feature_extractor = tf.keras.models.load_model(
        os.path.join(MODEL_DIR, 'feature_extractor.keras')
    )

assert (df_clean.index == np.arange(len(df_clean))).all(), "Index df_clean harus 0..N-1"
print("Dimensi vektor fitur:", feature_extractor.output_shape)
print(f"Galeri (df_clean): {len(df_clean)} | Query (test set): {len(test_df)}")

In [ ]:
FEAT_PATH = os.path.join(EVAL_DIR, 'features_all_model_aug.npy')

if os.path.exists(FEAT_PATH):
    features = np.load(FEAT_PATH)
    print("Fitur dimuat dari cache:", features.shape)
else:
    gallery_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
    all_generator = gallery_datagen.flow_from_dataframe(
        dataframe=df_clean,
        x_col='image_full_path',
        y_col=None,
        class_mode=None,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        validate_filenames=False,
    )

    features = feature_extractor.predict(all_generator, verbose=1)
    np.save(FEAT_PATH, features)
    print("Fitur disimpan:", features.shape)

In [ ]:
def precision_at_k(rel, k):
    return np.sum(rel[:k]) / k

def recall_at_k(rel, k, n_relevant_total):
    if n_relevant_total == 0:
        return 0.0

    return np.sum(rel[:k]) / n_relevant_total

def average_precision_at_k(rel, k, n_relevant_total=None):
    rel_k = rel[:k]

    if np.sum(rel_k) == 0:
        return 0.0

    precisions = [precision_at_k(rel_k, i + 1) for i in range(k) if rel_k[i] == 1]


    divisor = np.sum(rel_k)

    return np.sum(precisions) / divisor

In [ ]:
K_LIST   = [10]
MAX_K    = max(K_LIST)

categories   = df_clean['category'].values
query_idx    = test_df.index.values
cat_counts   = df_clean['category'].value_counts()

sim_matrix = cosine_similarity(features[query_idx], features)

records = []
for row, q in enumerate(tqdm(query_idx, desc="Evaluasi query")):
    sims = sim_matrix[row].copy()
    sims[q] = -np.inf

    #get the top 10 similarities
    top_idx = np.argsort(-sims)[:MAX_K]
    q_cat   = categories[q]
    rel     = (categories[top_idx] == q_cat).astype(int)
    n_rel   = cat_counts[q_cat] - 1

    rec = {'query_idx': q, 'category': q_cat, 'n_relevant_total': n_rel,
           'top_idx': top_idx.tolist(), 'rel': rel.tolist()}

    for k in K_LIST:
        rec[f'P@{k}']  = precision_at_k(rel, k)
        rec[f'R@{k}']  = recall_at_k(rel, k, n_rel)
        rec[f'AP@{k}'] = average_precision_at_k(rel, k, n_rel)

    records.append(rec)

eval_df = pd.DataFrame(records)
print(f"Jumlah query dievaluasi: {len(eval_df)}")

In [ ]:
summary = []

for k in K_LIST:
    summary.append({
        'K': k,
        f'Precision@K': eval_df[f'P@{k}'].mean(),
        f'Recall@K'   : eval_df[f'R@{k}'].mean(),
        f'MAP@K'      : eval_df[f'AP@{k}'].mean(),
    })

summary_df = pd.DataFrame(summary).set_index('K')
print("Hasil Evaluasi Retrieval (rata-rata seluruh query)")
print(summary_df.round(4))

summary_df.to_csv(os.path.join(EVAL_DIR, 'ringkasan_metrik.csv'))
eval_df.drop(columns=['top_idx', 'rel']).to_csv(os.path.join(EVAL_DIR, 'metrik_per_query.csv'), index=False)